# 05. Spearman Correlation Analysis

Phân tích tương quan **Feature ↔ Feature** trước Machine Learning.

**Nguyên tắc:** mỗi dòng trong `df` vẫn là một observation của dataset Feature Engineering. Spearman được tính trực tiếp trên các giá trị của từng feature, **không `groupBy`, không lấy trung bình theo cluster và không dùng `segment_profile`**.

Các feature có quan hệ toán học trực tiếp được loại khỏi nhánh Spearman để tập trung vào những mối quan hệ hành vi có ý nghĩa hơn.


## 5.1 Define Behavioral Feature Groups

Chỉ giữ các feature phù hợp để phân tích tương quan hành vi.

Loại khỏi Spearman các quan hệ cấu trúc/dẫn xuất trực tiếp:
- `late_delivery_count` ↔ `late_delivery_ratio`
- `negative_review_count` ↔ `negative_review_ratio`
- `total_items` ↔ `unique_products`
- `mean_matched_order_value` vì được dẫn xuất từ tổng giá trị và số đơn matched

Các feature theo khoảng thời gian như `orders_last_30d`, `orders_last_90d`, `orders_last_180d` vẫn được giữ vì chúng mô tả các mức độ hoạt động mua hàng khác nhau theo thời gian.


In [2]:
!pip install -q hvplot

import pandas as pd
import hvplot.pandas

from google.colab import drive
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

# Mount Google Drive
drive.mount(
    "/content/drive",
    force_remount=True
)

# ============================================================
# Load dataset
# ============================================================

DATA_PATH = (
    "/content/drive/MyDrive/BigData/DA-BigData/code/data/model_input/"
    "customer_churn_model_dataset.csv"
)

spark = (
    SparkSession.builder
    .appName("Olist-Spearman-Correlation")
    .getOrCreate()
)

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(DATA_PATH)
)

print("Using:", DATA_PATH)
print("Rows:", df.count())
print("Columns:", len(df.columns))

Mounted at /content/drive
Using: /content/drive/MyDrive/BigData/DA-BigData/code/data/model_input/customer_churn_model_dataset.csv
Rows: 213742
Columns: 36


In [3]:

RFM_FEATURES = [
    "recency_days",
    "frequency_orders",
    "monetary_matched_total"
]

PURCHASE_FEATURES = [
    "orders_last_30d",
    "orders_last_90d",
    "orders_last_180d"
]

DELIVERY_FEATURES = [
    "avg_delivery_days",
    "late_delivery_ratio"
]

REVIEW_FEATURES = [
    "review_count",
    "avg_review_score"
]

PRODUCT_FEATURES = [
    "total_items",
    "unique_categories",
    "avg_product_price"
]

FEATURE_GROUPS = {
    "RFM": RFM_FEATURES,
    "Purchase Behavior": PURCHASE_FEATURES,
    "Delivery": DELIVERY_FEATURES,
    "Review": REVIEW_FEATURES,
    "Product": PRODUCT_FEATURES
}

required_features = [
    feature
    for group in FEATURE_GROUPS.values()
    for feature in group
]

missing_features = sorted(
    set(required_features) - set(df.columns)
)

if missing_features:
    raise ValueError(
        f"Thiếu feature trong dataset: {missing_features}"
    )

print("Feature validation: OK")


Feature validation: OK


In [4]:
# Kiểm tra trực tiếp dữ liệu đầu vào.
# Không aggregate trước khi tính Spearman.

df.select(
    ["customer_unique_id", "snapshot_date"] + RFM_FEATURES
).show(5, truncate=False)


+--------------------------------+-------------+------------------+----------------+----------------------+
|customer_unique_id              |snapshot_date|recency_days      |frequency_orders|monetary_matched_total|
+--------------------------------+-------------+------------------+----------------+----------------------+
|01c228e91a729c993c054756eaeb680c|2018-02-01   |57.10399305555556 |1               |32.09                 |
|01f173d576ec0a01bbe13dccf0942696|2018-02-01   |121.57136574074076|1               |171.0                 |
|060e0b81f5dd707be0324b36479fffa6|2018-02-01   |162.0076388888889 |1               |176.65                |
|073b3547c939512360420845181cd3b7|2018-02-01   |61.57534722222222 |1               |124.01                |
|10ac9dd3c9585d66b06b38ce700a1b20|2018-02-01   |51.12903935185185 |1               |76.65                 |
+--------------------------------+-------------+------------------+----------------+----------------------+
only showing top 5 rows


## Hàm tính Spearman

`pyspark.ml.stat.Correlation` nhận trực tiếp các observation của từng feature và tính ma trận tương quan.

Không sử dụng giá trị trung bình của cluster.


In [5]:
def spearman_matrix(data, columns):
    clean = (
        data
        .select([
            F.col(c).cast("double").alias(c)
            for c in columns
        ])
        .na.drop()
    )

    vector_df = VectorAssembler(
        inputCols=columns,
        outputCol="features_vector"
    ).transform(clean)

    matrix = Correlation.corr(
        vector_df,
        "features_vector",
        method="spearman"
    ).head()[0]

    return pd.DataFrame(
        matrix.toArray(),
        index=columns,
        columns=columns
    )


## 5.2 RFM Correlation

Tập trung vào ba đặc trưng RFM:
- `recency_days`
- `frequency_orders`
- `monetary_matched_total`

Các cặp chính cần quan sát:
- Recency ↔ Frequency
- Frequency ↔ Monetary
- Recency ↔ Monetary


In [6]:
rfm_corr = spearman_matrix(
    df,
    RFM_FEATURES
)

rfm_corr.round(3)


,recency_days,frequency_orders,monetary_matched_total
recency_days,1.000,-0.007,0.012
frequency_orders,-0.007,1.000,0.150
monetary_matched_total,0.012,0.150,1.000


## 5.3 Purchase Behavior Correlation

So sánh các mức độ hoạt động mua hàng theo cửa sổ thời gian:

`30 ngày ↔ 90 ngày ↔ 180 ngày`.

Đây là các feature có liên hệ hành vi theo thời gian, nhưng không phải phép chia trực tiếp của nhau.


In [7]:
purchase_corr = spearman_matrix(
    df,
    PURCHASE_FEATURES
)

purchase_corr.round(3)


,orders_last_30d,orders_last_90d,orders_last_180d
orders_last_30d,1.000,0.365,0.015
orders_last_90d,0.365,1.000,0.095
orders_last_180d,0.015,0.095,1.000


## 5.4 Delivery Correlation

Chỉ giữ:

`avg_delivery_days` ↔ `late_delivery_ratio`

`late_delivery_count` và `delivery_order_count` không được đưa vào vì `late_delivery_ratio` được xây dựng từ các đại lượng này.


In [8]:
delivery_corr = spearman_matrix(
    df,
    DELIVERY_FEATURES
)

delivery_corr.round(3)


,avg_delivery_days,late_delivery_ratio
avg_delivery_days,1.000,0.319
late_delivery_ratio,0.319,1.000


## 5.5 Review Correlation

Chỉ giữ:

`review_count` ↔ `avg_review_score`

`negative_review_count` và `negative_review_ratio` được loại khỏi nhánh này vì chúng có quan hệ cấu trúc trực tiếp với nhau.


In [9]:
review_corr = spearman_matrix(
    df,
    REVIEW_FEATURES
)

review_corr.round(3)


,review_count,avg_review_score
review_count,1.000,-0.008
avg_review_score,-0.008,1.000


## 5.6 Product Correlation

Tập trung vào các đặc trưng không có quan hệ công thức trực tiếp:

- `total_items`
- `unique_categories`
- `avg_product_price`

`unique_products` được loại khỏi nhóm này để không lặp lại quan hệ cấu trúc với `total_items`.


In [10]:
product_corr = spearman_matrix(
    df,
    PRODUCT_FEATURES
)

product_corr.round(3)


,total_items,unique_categories,avg_product_price
total_items,1.000,0.282,-0.092
unique_categories,0.282,1.000,0.015
avg_product_price,-0.092,0.015,1.000


In [20]:
CORRELATION_THRESHOLD = 0.3

def extract_pairs(corr_df, group_name):
    pairs = []

    for i, feature_1 in enumerate(corr_df.index):
        for j, feature_2 in enumerate(corr_df.columns):
            if j <= i:
                continue

            rho = float(corr_df.iloc[i, j])

            pairs.append({
                "group": group_name,
                "feature_1": feature_1,
                "feature_2": feature_2,
                "spearman_rho": round(rho, 4)
            })

    return pairs


all_pairs = []

for group_name, columns in FEATURE_GROUPS.items():
    corr = spearman_matrix(df, columns)
    all_pairs.extend(
        extract_pairs(corr, group_name)
    )

all_pair_correlations = (
    pd.DataFrame(all_pairs)
    .sort_values(
        "spearman_rho",
        key=lambda s: s.abs(),
        ascending=False
    )
    .reset_index(drop=True)
)

notable_correlations = (
    all_pair_correlations[
        all_pair_correlations["spearman_rho"].abs()
        >= CORRELATION_THRESHOLD
    ]
    .reset_index(drop=True)
)

notable_correlations


,group,feature_1,feature_2,spearman_rho
0,Purchase Behavior,orders_last_30d,orders_last_90d,0.3646
1,Delivery,avg_delivery_days,late_delivery_ratio,0.3189


## 5.8 Visualize Correlation

Dùng **hvPlot** để xem heatmap theo từng nhóm và biểu đồ các cặp tương quan nổi bật.


In [21]:
def corr_heatmap(corr_df, title):
    plot_df = (
        corr_df
        .reset_index()
        .melt(
            id_vars="index",
            var_name="feature_2",
            value_name="spearman_rho"
        )
        .rename(columns={"index": "feature_1"})
    )

    return plot_df.hvplot.heatmap(
        x="feature_1",
        y="feature_2",
        C="spearman_rho",
        cmap="RdBu_r",
        clim=(-1, 1),
        colorbar=True,
        width=700,
        height=500,
        rot=45,
        title=title,
        tools=["hover"]
    )


In [13]:
corr_heatmap(
    rfm_corr,
    "RFM - Spearman Correlation"
)


:HeatMap   [feature_1,feature_2]   (spearman_rho)

In [14]:
corr_heatmap(
    purchase_corr,
    "Purchase Behavior - Spearman Correlation"
)


:HeatMap   [feature_1,feature_2]   (spearman_rho)

In [15]:
corr_heatmap(
    delivery_corr,
    "Delivery - Spearman Correlation"
)


:HeatMap   [feature_1,feature_2]   (spearman_rho)

In [16]:
corr_heatmap(
    review_corr,
    "Review - Spearman Correlation"
)


:HeatMap   [feature_1,feature_2]   (spearman_rho)

In [17]:
corr_heatmap(
    product_corr,
    "Product - Spearman Correlation"
)


:HeatMap   [feature_1,feature_2]   (spearman_rho)

In [18]:
if not notable_correlations.empty:
    plot_pairs = notable_correlations.copy()

    plot_pairs["pair"] = (
        plot_pairs["feature_1"]
        + " ↔ "
        + plot_pairs["feature_2"]
    )

    plot_pairs.hvplot.bar(
        x="pair",
        y="spearman_rho",
        by="group",
        rot=45,
        width=1000,
        height=500,
        title="Notable Spearman Correlations",
        xlabel="Feature Pair",
        ylabel="Spearman ρ",
        tools=["hover"]
    )


## 5.9 Interpretation & Take Notes

**Cách đọc kết quả:**

- `ρ > 0`: hai feature có xu hướng tăng cùng chiều.
- `ρ < 0`: hai feature có xu hướng thay đổi ngược chiều.
- `|ρ|` càng gần 1: quan hệ đơn điệu càng mạnh.
- Spearman chỉ phản ánh **tương quan**, không chứng minh quan hệ nhân quả.

**Nguyên tắc diễn giải của notebook:**
- Không diễn giải các feature được tạo từ cùng một công thức như một phát hiện hành vi độc lập.
- Tập trung vào các cặp có ý nghĩa hành vi và `|ρ| >= 0.50`.
- Kết quả Spearman không thay đổi `churn_180` và không dùng để thay thế Machine Learning.


In [22]:
print("=== ALL CANDIDATE PAIRS ===")
display(all_pair_correlations)

print("\n=== NOTABLE PAIRS (|rho| >= 0.50) ===")
display(notable_correlations)


=== ALL CANDIDATE PAIRS ===


,group,feature_1,feature_2,spearman_rho
0,Purchase Behavior,orders_last_30d,orders_last_90d,0.3646
1,Delivery,avg_delivery_days,late_delivery_ratio,0.3189
2,Product,total_items,unique_categories,0.2816
3,RFM,frequency_orders,monetary_matched_total,0.1500
4,Purchase Behavior,orders_last_90d,orders_last_180d,0.0951
5,Product,total_items,avg_product_price,-0.0917
6,Product,unique_categories,avg_product_price,0.0153
7,Purchase Behavior,orders_last_30d,orders_last_180d,0.0151
8,RFM,recency_days,monetary_matched_total,0.0120
9,Review,review_count,avg_review_score,-0.0078



=== NOTABLE PAIRS (|rho| >= 0.50) ===


,group,feature_1,feature_2,spearman_rho
0,Purchase Behavior,orders_last_30d,orders_last_90d,0.3646
1,Delivery,avg_delivery_days,late_delivery_ratio,0.3189


In [ ]:
OUTPUT_PATH = (
    "/content/drive/MyDrive/"
    "spearman_notable_correlations.csv"
)

notable_correlations.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Saved:", OUTPUT_PATH)


## 5.9 Take Notes

### Kết quả phân tích Spearman

Phân tích Spearman được thực hiện giữa các feature hành vi trong từng nhóm sau khi loại bỏ một số feature có quan hệ dẫn xuất hoặc cấu trúc trực tiếp, nhằm tránh việc kết quả tương quan chủ yếu phản ánh cách xây dựng feature.

Kết quả thu được:

- `orders_last_30d` và `orders_last_90d` có tương quan dương với `ρ = 0.3646`. Điều này cho thấy các customer-observation có số đơn trong 30 ngày gần nhất cao hơn có xu hướng có số đơn trong 90 ngày cao hơn. Tuy nhiên, mức tương quan không mạnh.
- `avg_delivery_days` và `late_delivery_ratio` có tương quan dương với `ρ = 0.3189`. Điều này cho thấy các customer-observation có thời gian giao hàng trung bình cao hơn có xu hướng có tỷ lệ giao hàng trễ cao hơn, nhưng mối quan hệ cũng không mạnh.
- `total_items` và `unique_categories` có tương quan dương với `ρ = 0.2816`, cho thấy số lượng item và độ đa dạng danh mục có xu hướng tăng cùng chiều nhưng mức liên hệ khá thấp.
- `frequency_orders` và `monetary_matched_total` có tương quan dương thấp với `ρ = 0.1500`.
- `recency_days` và `frequency_orders` gần như không có tương quan đơn điệu trong dữ liệu hiện tại với `ρ = -0.0066`.
- `review_count` và `avg_review_score` gần như không có tương quan với `ρ = -0.0078`.

### Kết luận

Sau khi loại các quan hệ có tính chất dẫn xuất trực tiếp, **không có cặp feature nào đạt mức tương quan mạnh với ngưỡng |ρ| ≥ 0.50** trong kết quả hiện tại.

Các mối quan hệ nổi bật nhất là giữa `orders_last_30d` với `orders_last_90d` (`ρ = 0.3646`) và giữa `avg_delivery_days` với `late_delivery_ratio` (`ρ = 0.3189`), nhưng cả hai chỉ thể hiện xu hướng dương ở mức thấp đến vừa.

Kết quả này cho thấy các feature được lựa chọn trong các nhóm RFM, Purchase Behavior, Delivery, Review và Product **không có xu hướng tương quan đơn điệu mạnh với nhau sau khi loại các quan hệ mang tính cấu trúc**.

Spearman trong bước này chỉ mô tả **mối quan hệ giữa các feature**, không dùng để kết luận feature nào gây ra feature nào và cũng không dùng để kết luận tác động đến `churn_180`.

Kết quả Spearman sẽ được sử dụng như một bước **khám phá mối quan hệ giữa các đặc trưng trước khi thực hiện các phân tích thống kê tiếp theo**.